# Mini Project: Smart Resume Search

## Setup
Installing the LangChain + PDF + vector store libraries.

In [ ]:
!pip install -q langchain-community pdfplumber langchain-text-splitters langchain-huggingface
!pip install -q langchain-chroma

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.document_loaders import PDFPlumberLoader
from langchain_chroma import Chroma

## Load PDF
Uploading a PDF and loading it into LangChain document objects.

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
docs = []

for filename in uploaded.keys():
    loader = PDFPlumberLoader(filename)
    docs.extend(loader.load())


In [ ]:
data = docs

## Chunking
Splitting the document into smaller overlapping text chunks.

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
chunks = text_splitter.split_documents(data)

print(chunks)

## Embeddings
Converting text chunks into vectors using a HuggingFace model.

In [ ]:
model_name = "sentence-transformers/all-mpnet-base-v2"
encoder = HuggingFaceEmbeddings(
    model_name=model_name
)

In [ ]:
string_chunks = [chunk.page_content for chunk in chunks]
encoder.embed_documents(string_chunks)

## Vector Store
Saving the embedded chunks into a persistent Chroma database.

In [ ]:
vector_db = Chroma.from_documents(
    documents=chunks,
    embedding=encoder,
    persist_directory="db",
    collection_name="vectorDB"
)

In [ ]:
vector_db = Chroma(
    embedding_function=encoder,
    persist_directory="db",
    collection_name="vectorDB"
)

## Query & Retrieve
Taking a user question and finding the most similar chunks.

In [ ]:
user_query = input("Enter user query: ")

In [ ]:
result = vector_db.similarity_search_with_score(query= user_query , k =3)
result

In [ ]:
import re

for doc, score in result:
    content = doc.page_content

    lines = content.strip().split('\n')
    name_or_header = lines[0] if lines else "Unknown"

    phone_match = re.search(r'(?:\+?\d{1,3}[-.\s]?)?\(?\d{3}\)?[-.\s]?\d{3}[-.\s]?\d{4}', content)
    mobile_no = phone_match.group(0) if phone_match else "No mobile number found"

    print(f"Name       : {name_or_header}")
    print(f"Mobile No. : {mobile_no}")
    print(f"Score      : {score}")
    print("-" * 50)